# 3 — Run Serving Experiments & Collect Metrics

This notebook builds and benchmarks five serving options on Chameleon:

| # | Option | Backend | Workers | Hardware |
|---|--------|---------|---------|----------|
| 1 | `baseline` | Mock (no model) | 1 uvicorn | CPU |
| 2 | `torch_cpu` | PyTorch `.pt` | 1 uvicorn | CPU |
| 3 | `onnx_cpu` | ONNX Runtime | 1 uvicorn | CPU |
| 4 | `onnx_multiworker` | ONNX Runtime | 4 gunicorn | CPU |
| 5 | `torch_gpu` | PyTorch `.pt` | 1 uvicorn | GPU (RTX 6000) |

**Prerequisites:** notebook 2 completed, server is up, repo is cloned.

## Configuration

In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease
import os, time, json

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@UC")

In [ ]:
# runs in Chameleon Jupyter environment
NET_ID     = "YOUR_NET_ID"       # ← same as notebooks 1 & 2
LEASE_NAME = f"serve_jellyfin_{NET_ID}"
REPO_DIR   = "~/jellyfin-recommender"

# Benchmark parameters
TOTAL_REQUESTS = 500
CONCURRENCY_CPU = 1    # single-thread to measure per-request latency
CONCURRENCY_MW  = 8    # higher concurrency to stress the multi-worker option
CONCURRENCY_GPU = 4

l = lease.get_lease(LEASE_NAME)
username = os.getenv('USER')
s = server.Server.from_name(f"node-serve-jellyfin-{username}")
s.refresh()
FLOATING_IP = s.floating_ip
print(f"Server floating IP: {FLOATING_IP}")

## Helper: build image, run container, benchmark, stop

In [ ]:
# runs in Chameleon Jupyter environment

def build_image(s, context_dir: str, tag: str):
    print(f"Building image: {tag} from {context_dir}")
    stdout, rc = s.execute(f"docker build -t {tag} {context_dir}")
    if rc != 0:
        raise RuntimeError(f"Build failed:\n{stdout}")
    print(f"  Built: {tag}")


def start_container(s, tag: str, name: str, extra_args: str = ""):
    # Remove any existing container with the same name
    s.execute(f"docker rm -f {name} 2>/dev/null || true")
    cmd = f"docker run -d --rm -p 8000:8000 --name {name} {extra_args} {tag}"
    stdout, rc = s.execute(cmd)
    if rc != 0:
        raise RuntimeError(f"docker run failed:\n{stdout}")
    print(f"  Container started: {name}")


def wait_healthy(s, name: str, timeout: int = 120):
    print(f"  Waiting for {name} to be healthy...", end="", flush=True)
    for _ in range(timeout // 5):
        out, _ = s.execute(f"docker inspect --format='{{{{.State.Health.Status}}}}' {name}")
        status = out.strip().strip("'")
        if status == "healthy":
            print(" healthy.")
            return
        print(".", end="", flush=True)
        time.sleep(5)
    raise TimeoutError(f"{name} did not become healthy within {timeout}s")


def run_benchmark(s, floating_ip: str, total: int, concurrency: int) -> dict:
    cmd = (
        f"cd {REPO_DIR} && "
        f"BASE_URL=http://localhost:8000 "
        f"TOTAL_REQUESTS={total} CONCURRENCY={concurrency} "
        f"python3 -m benchmarks.benchmark_recommend"
    )
    stdout, rc = s.execute(cmd)
    print(stdout)
    # Parse key numbers from stdout
    metrics = {}
    for line in stdout.splitlines():
        for key, label in [
            ("Throughput", "throughput_rps"),
            ("Latency p50", "p50_ms"),
            ("Latency p95", "p95_ms"),
            ("Success rate", "success_rate"),
        ]:
            if key in line:
                val_str = line.split(":")[-1].strip().split()[0].rstrip("%")
                try:
                    metrics[label] = float(val_str)
                except ValueError:
                    pass
    return metrics


def stop_container(s, name: str):
    s.execute(f"docker rm -f {name} 2>/dev/null || true")
    print(f"  Stopped: {name}")


results = {}  # will accumulate per-option metrics
print("Helpers loaded.")

---
## Option 1: baseline (mock model, 1 CPU worker)

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "baseline"
TAG  = f"jellyfin-{OPT}"
build_image(s, f"{REPO_DIR}/serving/baseline", TAG)
start_container(s, TAG, f"container-{OPT}")
wait_healthy(s, f"container-{OPT}")

metrics = run_benchmark(s, FLOATING_IP, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_CPU)
results[OPT] = metrics

stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] metrics: {metrics}")

---
## Option 2: torch_cpu (PyTorch .pt, 1 CPU worker)

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "torch_cpu"
TAG  = "jellyfin-torch-cpu"
build_image(s, f"{REPO_DIR}/serving/torch_model", TAG)
start_container(s, TAG, f"container-{OPT}")
wait_healthy(s, f"container-{OPT}")

metrics = run_benchmark(s, FLOATING_IP, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_CPU)
results[OPT] = metrics

stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] metrics: {metrics}")

---
## Option 3: onnx_cpu (ONNX Runtime, 1 CPU worker — model-level optimisation)

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "onnx_cpu"
TAG  = "jellyfin-onnx-cpu"
build_image(s, f"{REPO_DIR}/serving/onnx", TAG)
start_container(s, TAG, f"container-{OPT}")
wait_healthy(s, f"container-{OPT}")

metrics = run_benchmark(s, FLOATING_IP, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_CPU)
results[OPT] = metrics

stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] metrics: {metrics}")

---
## Option 4: onnx_multiworker (ONNX Runtime, 4 gunicorn workers — system-level optimisation)

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "onnx_multiworker"
TAG  = "jellyfin-onnx-mw"
build_image(s, f"{REPO_DIR}/serving/multiworker", TAG)
start_container(s, TAG, f"container-{OPT}")
wait_healthy(s, f"container-{OPT}")

# Higher concurrency to actually stress the extra workers
metrics = run_benchmark(s, FLOATING_IP, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_MW)
results[OPT] = metrics

stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] metrics: {metrics}")

---
## Option 5: torch_gpu (PyTorch .pt, 1 uvicorn worker, RTX 6000 GPU — hardware-level optimisation)

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "torch_gpu"
TAG  = "jellyfin-torch-gpu"
build_image(s, f"{REPO_DIR}/serving/torch_model_gpu", TAG)
start_container(s, TAG, f"container-{OPT}", extra_args="--gpus all")
wait_healthy(s, f"container-{OPT}")

metrics = run_benchmark(s, FLOATING_IP, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_GPU)
results[OPT] = metrics

stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] metrics: {metrics}")

---
## Results summary table

Fill in the **Endpoint URL**, **Model version**, **Code version (git sha)**, and **Hardware** columns manually from the cell outputs above, then include this table in your Gradescope PDF.

In [ ]:
# runs in Chameleon Jupyter environment
import pandas as pd

rows = []
for opt, m in results.items():
    rows.append({
        "Option": opt,
        "p50 latency (ms)": m.get("p50_ms", "—"),
        "p95 latency (ms)": m.get("p95_ms", "—"),
        "Throughput (req/s)": m.get("throughput_rps", "—"),
        "Success rate": m.get("success_rate", "—"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## Check resource usage (right-sizing)

Re-start your most promising option, then use `docker stats` to capture CPU/memory/GPU under representative load.

In [ ]:
# runs in Chameleon Jupyter environment
# Example: measure resource usage for the best option under load
# Change BEST_OPT and BEST_TAG to whichever option you are right-sizing

BEST_OPT = "onnx_multiworker"   # ← change as needed
BEST_TAG = "jellyfin-onnx-mw"   # ← matching image tag
BEST_EXTRA = ""                  # ← "--gpus all" for GPU option

start_container(s, BEST_TAG, f"container-best", extra_args=BEST_EXTRA)
wait_healthy(s, "container-best")

# Send load in background, snapshot stats once
s.execute(
    f"cd {REPO_DIR} && "
    f"BASE_URL=http://localhost:8000 TOTAL_REQUESTS=200 CONCURRENCY=8 "
    f"python3 -m benchmarks.benchmark_recommend &"
)
time.sleep(5)   # let load build up

stdout, _ = s.execute("docker stats container-best --no-stream")
print(stdout)

stop_container(s, "container-best")

In [ ]:
# runs in Chameleon Jupyter environment
# GPU utilisation for the GPU option
stdout, _ = s.execute("nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv")
print(stdout)

---
**Done.** Copy the results table and `docker stats` / `nvidia-smi` output into your Gradescope PDF submission.